# 🔄 Transformación Medallion: Bronze → Silver → Gold

Este notebook implementa el flujo de transformación de datos a través de las tres capas de la arquitectura Medallion en Microsoft Fabric.

**Flujo:**
1. Leer datos crudos desde **Lakehouse Bronze**
2. Limpiar y estandarizar → escribir en **Lakehouse Silver**
3. Agregar y modelar → escribir en **Lakehouse Gold**

> ⚠️ Asegúrate de haber agregado los tres Lakehouses (Bronze, Silver y Gold) como recursos de este Notebook antes de ejecutarlo.

## ⚙️ Parámetros de configuración

Define aquí los nombres de los Lakehouses y las tablas a procesar.

In [ ]:
# Nombres de los Lakehouses configurados en este Notebook
LAKEHOUSE_BRONZE = "Lakehouse_Bronze"
LAKEHOUSE_SILVER = "Lakehouse_Silver"
LAKEHOUSE_GOLD   = "Lakehouse_Gold"

# Nombre de la tabla fuente en Bronze (ajustar según el desafío)
TABLA_BRONZE = "sqls_nombre_tabla"

# Nombres de las tablas destino
TABLA_SILVER = "nombre_tabla"
TABLA_GOLD   = "nombre_tabla_agg"

---
## 🟤 Paso 1 – Lectura desde Bronze

Leemos la Delta Table del Lakehouse Bronze sin aplicar ninguna modificación.

In [ ]:
df_bronze = spark.read.format("delta").load(
    f"abfss://{LAKEHOUSE_BRONZE}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_BRONZE}.Lakehouse/Tables/{TABLA_BRONZE}"
)

print(f"Registros leídos desde Bronze: {df_bronze.count()}")
df_bronze.printSchema()
df_bronze.show(5, truncate=False)

---
## 🔵 Paso 2 – Limpieza y estandarización → Silver

Aplicamos las siguientes transformaciones:
- Eliminación de filas duplicadas
- Tratamiento de valores nulos
- Corrección de tipos de datos
- Estandarización de textos y fechas

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType, DoubleType, IntegerType

df_silver = (
    df_bronze
    # 1. Eliminar duplicados
    .dropDuplicates()

    # 2. Eliminar filas donde la clave primaria sea nula
    # AJUSTAR: reemplazar 'id' por el nombre real de la clave primaria
    .filter(F.col("id").isNotNull())

    # 3. Estandarizar textos: trim y lowercase en columnas de tipo string
    # AJUSTAR: reemplazar 'nombre' y 'categoria' por las columnas reales
    .withColumn("nombre",    F.trim(F.lower(F.col("nombre"))))
    .withColumn("categoria", F.trim(F.lower(F.col("categoria"))))

    # 4. Castear tipos de datos
    # AJUSTAR: adaptar según el esquema real de la tabla
    .withColumn("fecha",  F.col("fecha").cast(TimestampType()))
    .withColumn("monto",  F.col("monto").cast(DoubleType()))
    .withColumn("cantidad", F.col("cantidad").cast(IntegerType()))

    # 5. Reemplazar nulos en columnas numéricas con 0
    .fillna({"monto": 0.0, "cantidad": 0})

    # 6. Agregar columna de auditoría
    .withColumn("_fecha_carga", F.current_timestamp())
)

print(f"Registros después de limpieza: {df_silver.count()}")
df_silver.show(5, truncate=False)

In [ ]:
# Escritura en Lakehouse Silver como Delta Table
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(
        f"abfss://{LAKEHOUSE_SILVER}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_SILVER}.Lakehouse/Tables/{TABLA_SILVER}"
    )
)

print(f"✅ Datos escritos en Silver: {LAKEHOUSE_SILVER}/Tables/{TABLA_SILVER}")

---
## 🟡 Paso 3 – Agregación y modelado → Gold

Leemos desde Silver y aplicamos agregaciones para generar una tabla lista para consumo analítico.

In [ ]:
df_silver_read = spark.read.format("delta").load(
    f"abfss://{LAKEHOUSE_SILVER}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_SILVER}.Lakehouse/Tables/{TABLA_SILVER}"
)

# AJUSTAR: adaptar las columnas de agrupación y métricas según el modelo de negocio
df_gold = (
    df_silver_read
    .groupBy("categoria", F.to_date("fecha").alias("fecha"))
    .agg(
        F.sum("monto").alias("total_monto"),
        F.sum("cantidad").alias("total_cantidad"),
        F.countDistinct("id").alias("total_registros"),
        F.avg("monto").alias("promedio_monto")
    )
    .withColumn("_fecha_carga", F.current_timestamp())
)

print(f"Registros en Gold: {df_gold.count()}")
df_gold.show(10, truncate=False)

In [ ]:
# Escritura en Lakehouse Gold como Delta Table
(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(
        f"abfss://{LAKEHOUSE_GOLD}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_GOLD}.Lakehouse/Tables/{TABLA_GOLD}"
    )
)

print(f"✅ Datos escritos en Gold: {LAKEHOUSE_GOLD}/Tables/{TABLA_GOLD}")

---
## ✅ Verificación final

Validamos que los datos existen correctamente en Silver y Gold.

In [ ]:
print("=== RESUMEN DE CARGA ===")
print(f"Bronze  → {df_bronze.count():,} registros (sin modificar)")
print(f"Silver  → {df_silver.count():,} registros (limpios)")
print(f"Gold    → {df_gold.count():,} registros (agregados)")